In [113]:
#https://huggingface.co/siebert/sentiment-roberta-large-english
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("siebert/sentiment-roberta-large-english")

model = AutoModelForSequenceClassification.from_pretrained("siebert/sentiment-roberta-large-english")

In [2]:
#Electronics Dataset:

import numpy as np
import pandas as pd
fileElectr='amazon_reviews_us_Electronics_v1_00.tsv'
df=pd.read_csv(fileElectr, sep="\t", header=0, on_bad_lines='skip')
df=df.dropna(subset=['review_headline', 'review_body', 'star_rating'])

In [105]:
n_samples=1000

N_rewiews=df.loc[df['star_rating'] == 1].sample(n_samples, replace=False, random_state=1900)
N_rewiew2=df.loc[df['star_rating'] == 5].sample(n_samples, replace=False, random_state=1900)


samplesize=n_samples*2

N_rewiews=N_rewiews.append(N_rewiew2)


/var/folders/ds/k83592y50w34wqwchx8qldph0000gn/T/ipykernel_35741/1444290911.py:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  N_rewiews=N_rewiews.append(N_rewiew2)


In [106]:
N_rewiews=N_rewiews.reset_index()

In [7]:
from transformers import pipeline


In [66]:
text=N_rewiews['review_body'][4]  
sentiment_analysis = pipeline("sentiment-analysis",model="siebert/sentiment-roberta-large-english")

print(sentiment_analysis(text))



[{'label': 'NEGATIVE', 'score': 0.9995100498199463}]


In [65]:
N_rewiews['review_body'][4]

'These are not decent. This earphone set is split into two cords. 2ft to an L bent connector, to another 2 ft extension male/female cord. This is the worst design ever. They meant for you to be able to disconnect the cord in half for convenience, as if you were to wear your device as a necklace.<br /><br />The fact that the two cords connect at an L bent configuration is immensely dysfunctional. It gets caught in your clothes, and all of your stuff on your desk. I have to throw these away. WOw, such bad design. much refund never'

In [43]:
text=N_rewiews['review_body'][1400]
sentiment_analysis = pipeline("sentiment-analysis",model="siebert/sentiment-roberta-large-english")
print(sentiment_analysis(text))



[{'label': 'POSITIVE', 'score': 0.9983807802200317}]


In [44]:
out=sentiment_analysis(text)

In [45]:
out[0]['label']

'POSITIVE'

In [46]:
out[0]['label']=='POSITIVE'

True

In [47]:
print(text)


Allowed me to use my Plantronics headset with my Samsung Galaxy SIII as well as with my work laptop, a Dell Latitude E5430.


In [49]:
N_rewiews['star_rating'][1400]

5

In [34]:
N_rewiews.columns

Index(['index', 'marketplace', 'customer_id', 'review_id', 'product_id',
       'product_parent', 'product_title', 'product_category', 'star_rating',
       'helpful_votes', 'total_votes', 'vine', 'verified_purchase',
       'review_headline', 'review_body', 'review_date'],
      dtype='object')

In [41]:
N_rewiews['star_rating'].value_counts()

1    1000
5    1000
Name: star_rating, dtype: int64

In [55]:
import time
t = time.time()

out=sentiment_analysis(N_rewiews['review_body'][21])

elapsed = time.time() - t

print(elapsed)

1.0142948627471924


In [103]:
from transformers import pipeline
#print(sentiment_analysis("I love this!"))
    
    
def sentiment_classify(df_sample, column_name):
    sentiment_analysis = pipeline("sentiment-analysis",model="siebert/sentiment-roberta-large-english")


    for i in range (0, len(df_sample[column_name])):

        #processed text
        text=df_sample[column_name][i] 
        #print(text, "We are at i=", str(i)")
        
        
        
        if len(text) > 514:
            text = text[:514]
            #RuntimeError: The expanded size of the tensor (536) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 536].  Tensor sizes: [1, 514]

            
            
        prediction={}
        prediction = sentiment_analysis(text)

        if prediction[0]['label']=='POSITIVE':
            df_sample.loc[i, ("sentiment_analysis")]=5


        elif prediction[0]['label']== 'NEGATIVE':
            df_sample.loc[i, ("sentiment_analysis")]=1
        else: 
            print("Error.")
            
        if i%1000==0:
           print ("\n sentiment_classify:   We are at i=", str(i))    
            
            
    return df_sample
            


In [107]:
t = time.time()

sentiment_classify(N_rewiews, 'review_body')

elapsed = time.time() - t

print(elapsed)



 sentiment_classify:   We are at i= 0

 sentiment_classify:   We are at i= 1000
2749.1342618465424


In [109]:
N_rewiews

,index,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date,sentiment_analysis
0,1677873,US,18585476,R31LJGNJRWGRS,B003ARSOWQ,864558418,Timex T715BW3 Dual Alarm Clock Radio (Black),Electronics,1,0,1,N,Y,"Does it keep time, not really.","I have had the product for just under a month,...",2013-12-14,1.0
1,1856468,US,51075252,R35YF0DWJE87A4,B0044WS7KK,471031907,"Aerial7 Perisher - Black - black, one size",Electronics,1,0,1,N,Y,Very Poor Sound,The speaker quality reminds you of a 1960's tr...,2013-08-12,1.0
2,600712,US,15700552,R2PQJHUI1SF24E,B00INO6JX2,703104763,Samsung SSG-5150GB 3D Active Glasses,Electronics,1,1,3,N,Y,Wrong item,Did not work,2015-02-24,1.0
3,2871105,US,26397372,R1B30LZ8X5V07Q,B00008VSK5,50332,Acoustic Research MS805 Adaptatip Flex Pin,Electronics,1,4,5,N,Y,"Flex pin set incomplete, not worth it.",This flex pin set requires an additional part:...,2008-12-22,1.0
4,1381712,US,20903669,R2B9JFV8C5AHX,B007N16IYG,623454097,Panasonic Deep Base Ergo-Fit Inner Ear Earbud ...,Electronics,1,1,4,N,Y,These are not what you think. Much regrets,These are not decent. This earphone set is spl...,2014-05-16,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1501310,US,34583004,R3DI5BWSS2VD5W,B005HXFARS,120807590,Ports HDMI Powered Splitter for Full HD,Electronics,5,0,0,N,Y,Five Stars,Item as described and shipped fast !!,2014-03-06,5.0
1996,2920100,US,13725156,R3DFLPX0FJVII2,B000BTHHAG,483950545,Remanufactured Panasonic CQ-VD7001U DVD/CD Rec...,Electronics,5,0,2,N,Y,Panasonic CQ-VD7001U DVD/CD,Unit does what it says looks sharp with no pro...,2008-03-29,5.0
1997,2997394,US,20832992,R2DKC5UBA0SRKT,B000IF4TPY,623089856,Creative Zen Vision W 30 GB Widescreen Multime...,Electronics,5,2,3,N,Y,Santa's Perfect Present,"I bought this for my wife - who is not an \\""e...",2007-01-09,5.0
1998,3090837,US,52667938,R5R2WFA7N0TMH,B00000J4IT,667909553,TDK Recordable Minidisc (5-Pack),Electronics,5,8,8,N,N,"Good product, low price!","First minidiscs I bought, most offer 74 minute...",1999-09-18,5.0


In [110]:
N_rewiews['sentiment_analysis'].value_counts()

5.0    1005
1.0     995
Name: sentiment_analysis, dtype: int64

In [111]:
#nothing to train here so we can directly go to the evaluation step:
y_pred= N_rewiews['sentiment_analysis']
y_test=N_rewiews['star_rating']

In [112]:
from sklearn.metrics import classification_report

target_names = ['0 = rating of 1',  '1 = rating of 5'] # 0 = negative, 4 = positive

print(classification_report(y_test, y_pred, target_names=target_names, digits=6))



                 precision    recall  f1-score   support

0 = rating of 1   0.973869  0.969000  0.971429      1000
1 = rating of 5   0.969154  0.974000  0.971571      1000

       accuracy                       0.971500      2000
      macro avg   0.971512  0.971500  0.971500      2000
   weighted avg   0.971512  0.971500  0.971500      2000

